Transformer: Encoder + Decoder

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import numpy as np
from collections.abc import Callable

In [2]:
# input/output embedding + positional embedding
class Embeddings(nn.Module):
    def __init__(self, vocab, d_model):
        super(Embeddings, self).__init__()
        # learnable lookup table shape (vocab, d_model)
        self.embedding = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        # x shape (sequences, seq_len)
        # embeddings shape (sequences, seq_len, d_model)
        return self.embedding(x) * math.sqrt(self.d_model)

class PositionalEncoder(nn.Module):
    """Positional Encoding for transformer model.
    """
    def __init__(self, device, d_model, seq_len, dropout):
        super(PositionalEncoder, self).__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        # Create positional encoding matrix (seq_len, d_model)
        # Even dimensions (i = 0, 2, 4, ...): PE(pos, i) = sin(pos / 10000^(i/d_model))
        # Odd dimensions (i = 1, 3, 5, ...): PE(pos, i) = cos(pos / 10000^(i/d_model))
        pe = torch.zeros(seq_len, d_model, device=device)
        print(f"pe initial shape: {pe.shape}")
        
        # Create the position matrix (seq_len, 1)
        position = torch.arange(0, seq_len, device=device).unsqueeze(1)
        print(f"position shape: {position.shape}")
        # Create the div_term matrix (d_model // 2,)
        # div_term_org = 1.0 / (10000.0 ** (torch.arange(0., d_model, 2, device=device).float() / d_model))
        div_term = torch.exp(torch.arange(0., d_model, 2, device=device) * - (math.log(10000.0) / d_model)).unsqueeze(0)
        # if torch.allclose(div_term_org, div_term):
        #     print(f"identical div_term")
        # else:
        #     print(f"div_terms are different")
        #     print(f"Max difference: {(div_term_org - div_term).abs().max()}")
        print(f"div_term shape: {div_term.shape}")
        # Create the pe_pos matrix
        # pe_pos = position.float() * div_term
        # pe_pos shape: (seq_len, d_model // 2)
        pe_pos = torch.mul(position, div_term)
        print(f"pe_pos shape: {pe_pos.shape}")

        # Apply sin to even indices
        pe[:, 0::2] = torch.sin(pe_pos)
        # Apply cos to odd indices
        pe[:, 1::2] = torch.cos(pe_pos)

        pe = pe.unsqueeze(0)
        print(f"pe final shape: {pe.shape}")
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        print(f"[PE]: x shape: {x.shape}")
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [3]:
# Common for Encoder and Decoder
def attention(query, key, value, mask=None, dropout=None):
    """Scaled Dot-Product Attention
                     ▲
        ┌────────────┴─────────────┐
        │4) MatMul                 │ ← Attention Weights · V
        └────▲──────────────────▲──┘
        ┌────┴──────┐           │  
        │3) Softmax │           │
        └────▲──────┘           │
       ┌─────┴────────┐         │
       │2) Mask (opt.)│         │
       └─────▲────────┘         │
        ┌────┴──────┐           │
        │1) Scale   │ (/ √d_k)  │
        └────▲──────┘           │
        ┌────┴──────┐           │
        │1) MatMul  │ ← Q · Kᵀ  │
        └────▲──────┘           │
       ┌─────┴─────┐────────────│ 
       Q           K            V
    """
    print(f"+AT: q/k/v shapes: {query.shape}")
    d_k = query.size(-1)
    # input: query, key, value shape: (batch_size, h, seq_len, d_k)
    #
    # 1) MatMul(QK^T) + Scale(math.sqrt(d_k)) (scaled dot-product):
    #    (batch_size, h, seq_len, d_k) - > (batch_size, h, seq_len, seq_len)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    #
    # 2) Mask optional for decoder
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    print(f"\tscores shape: {scores.shape}")
    #
    # 3) softmax (batch_size, h, seq_len, seq_len) -> (batch_size, h, seq_len, seq_len)
    attn = F.softmax(scores, dim=-1)
    print(f"\tattn shape: {scores.shape}")
    if dropout is not None:
        attn = dropout(attn)
    #
    # 4) MatMul(AV) (batch_size, h, seq_len, seq_len) * (batch_size, h, seq_len, d_k) -> (batch_size, h, seq_len, d_k)
    #    (d_v == d_k == d_model // h)
    out = torch.matmul(attn, value)    
    print(f"-AT: output shape:{out.shape}")
    return out, attn

class MultiHeadAttention(nn.Module):
    """Multi-head attention: d_model dimensional, h heads
                           ┌─────────┐
                           │ Linear  │
                           └────▲────┘
                                │
                           ┌────┴────┐
                           │ Concat  │
                           └────▲────┘
                                │
        ┌─────────────────────────────────────────┐
        │     Scaled Dot-Product Attention        │   × h
        └─────────────────────────────────────────┘
             ▲                ▲                ▲
             │                │                │
         ┌───┴───┐        ┌───┴───┐        ┌───┴───┐
         │ Linear│        │ Linear│        │ Linear│
         └───▲───┘        └───▲───┘        └───▲───┘
             │                │                │
             V                K                Q
    """
    def __init__(self, d_model, h, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        # 4 linear layers: Q, K, V projections + output projection
        self.linears = nn.ModuleList([nn.Linear(in_features=d_model, out_features=d_model) for _ in range(4)])
        self.dropout = nn.Dropout(p=dropout)
        # store the probability score for visualization
        self.attn = None
    
    def forward(self, query, key, value, mask=None):
        print(f"+MHA:{list(query.shape)}")
        # query, key, value shape (batch_size, seq_len, d_model)
        batch_size = query.size(0)
        # 1) Linear projections
        #    q_p, k_p, v_p shape (batch_size, h, seq_len, d_k)
        q_p, k_p, v_p = [l(x).view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
                        for l, x in zip(self.linears, (query, key, value))]
        # 2) h times attention (in parallel)
        # concat_attn shape (batch_size, h, seq_len, d_k)
        concat_attn, self.attn = attention(q_p, k_p, v_p, mask, self.dropout)
        # 3) Concat
        # reshape to (batch_size, seq_len, h, d_k)
        concat_attn = concat_attn.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d_k)
        # concat to (batch_size, seq_len, d_model(h * d_k))
        # concat_attn = concat_attn.view(batch_size, -1, self.h * self.d_k)
        # 4) linear
        # out shape (batch_size, seq_len, d_model)
        out = self.linears[-1](concat_attn)
        print(f"-MHA: output shape:{list(out.shape)}")
        return out

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.linear_1 = nn.Linear(in_features=d_model, out_features=d_ff) # w_1 and b_1
        self.linear_2 = nn.Linear(in_features=d_ff, out_features=d_model) # w_2 and b_2
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        print(f"+FFN: x.shape:{x.shape}")
        h1 = self.linear_1(x)
        h2 = self.dropout(torch.relu(h1))
        out_ffn = self.linear_2(h2)
        print(f"-FFN:out shape:{out_ffn.shape}")
        return out_ffn

# Layer normalization
class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(features)) # alpha is a learnable parameter
        self.bias = nn.Parameter(torch.zeros(features)) # bias is a learnable parameter

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        # mean shape: (batch, seq_len, 1)
        print(f"+LN x shape:{x.shape}")
        mean = x.mean(-1, keepdim=True)
        print(f"\tLN: mean shape: {mean.shape}")
        # std shape: (batch, seq_len, 1)
        std = x.std(-1, keepdim=True, unbiased=False)
        print(f"\tLN: std shape: {mean.shape}")
        output_n = self.weight * (x - mean) / (std + self.eps) + self.bias
        print(f"-LN output shape:{output_n.shape}")
        return output_n

class ResidualConnection(nn.Module):
    def __init__(self, features: int, dropout: float = 0.1):
        super(ResidualConnection, self).__init__()
        self.norm = LayerNorm(features)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer: Callable):
        print(f"+RC: x.shape={x.shape}")
        # out_rc = self.norm(x + self.dropout(sublayer(x)))
        # POST-NORM vs PRE-NORM: 
        # This implementation uses PRE-NORM (norm before sublayer)
        # but the original transform paper used POST-NORM (norm after addition):
        #     Input (x)
        #    |
        #    ├─────────────────┐
        #    |                 |
        #    |            Sublayer(x)
        #    |                 |
        #    |             Dropout
        #    |                 |
        #    └────── (+) ──────┘
        #           |
        #      LayerNorm
        #           |
        #        Output
        # LayerNorm(x + Sublayer(x))
        # out_rc = self.norm(x + self.dropout(sublayer(x)))
        #
        # Modern Implementation: Pre-Norm (LayerNorm BEFORE sublayer)
        # often works better in practice
        # x + Sublayer(LayerNorm(x))
        #
        #     Input (x)
        #    |
        #    ├─────────────────┐
        #    |                 |
        #    |            LayerNorm
        #    |                 |
        #    |            Sublayer
        #    |                 |
        #    |             Dropout
        #    |                 |
        #    └────── (+) ──────┘
        #           |
        #        Output
        #
        # ================================================================================
        # KEY DIFFERENCES
        # ================================================================================

        # Aspect                    Post-Norm (Paper)              Pre-Norm (Modern)             
        # --------------------------------------------------------------------------------
        # Normalization Position    After residual addition        Before sublayer               
        # Gradient Flow             Through normalization          Direct path via residual      
        # Training Stability        Can be unstable                More stable                   
        # Learning Rate             Needs warmup                   Less sensitive                
        # Convergence               Slower initially               Faster convergence            
        # Final Performance         Slightly better (sometimes)    Slightly worse (sometimes)    
        # Ease of Training          Harder                         Easier                        

        out_rc = x + self.dropout(sublayer(self.norm(x)))
        print(f"-RC: output shape: {out_rc.shape}")
        return out_rc

In [4]:
# Encoder
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, self_attention: MultiHeadAttention, feed_forward: PositionwiseFeedForward, dropout: float):
        super(EncoderLayer, self).__init__()
        self.self_attention = self_attention
        self.feed_forward = feed_forward
        self.sublayers = nn.ModuleList([ResidualConnection(d_model, dropout) for _ in range(2)])
        self.d_model = d_model

    def forward(self, x, mask):
        print(f"+EN_LAYER: x shape:{x.shape} mask shape: {mask.shape if mask is not None else None}")
        x = self.sublayers[0](x, lambda x: self.self_attention(x, x, x, mask))
        # x = self.sublayers[1](x, self.feed_forward)
        x = self.sublayers[1](x, lambda x: self.feed_forward(x))
        print(f"-EN_LAYER: output shape:{x.shape}")
        return x

class Encoder(nn.Module):
    """Encoder stack with N identical layers."""
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = LayerNorm(layer.d_model)

    def forward(self, x, mask):
        """Pass the input (and mask) through each layer in turn."""
        print(f"+ENCODER: x.shape={x.shape}, mask.shape={mask.shape if mask is not None else None}")
        for layer in self.layers:
            x = layer(x, mask)
        out_encoder = self.norm(x)
        print(f"-ENCODER: output.shape={out_encoder.shape}")
        return out_encoder

In [18]:
# Decoder
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, self_attn, cross_attn, ffn, dropout):
        super(DecoderLayer, self).__init__()
        self.d_model = d_model
        self.self_attn = self_attn
        self.cross_attn = cross_attn
        self.ffn = ffn
        self.sublayers = nn.ModuleList([ResidualConnection(d_model, dropout) for _ in range(3)])
    
    def forward(self, tgt, memory, tgt_mask, src_mask):
        print(f"+DC_LAYER:x.shape={tgt.shape}, memory.shape={memory.shape}")
        tgt = self.sublayers[0](tgt, lambda tgt: self.self_attn(tgt, tgt, tgt, tgt_mask))
        tgt = self.sublayers[1](tgt, lambda tgt: self.cross_attn(tgt, memory, memory, src_mask))
        tgt = self.sublayers[2](tgt, self.ffn)
        print(f"-DC_LAYER:output.shape={tgt.shape}")
        return tgt

class Decoder(nn.Module):
    def __init__(self, layer, N):
        """Decoder stack with N layers"""
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
        self.norm = LayerNorm(layer.d_model)

    def forward(self, tgt, memory, tgt_mask, src_mask):
        print(f"+DECODER:x shape:{tgt.shape} memory shape:{memory.shape} tgt_mask shape:{tgt_mask.shape if tgt_mask is not None else 'None'} src_mask shape:{src_mask.shape}")
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, src_mask)
        output_decoder = self.norm(tgt)
        print(f"-DECODER:output shape:{output_decoder.shape}")
        return output_decoder

In [ ]:
class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(in_features=d_model, out_features=vocab)
    
    def forward(self, x):
        print(f"+GENERATOR:x.shape={x.shape}")
        proj_x = self.proj(x)
        g_out = F.log_softmax(proj_x, dim=-1)
        print(f"-GENERATOR:g_out.shape={g_out.shape}")
        return g_out

In [ ]:
# Transformer

class Transformer(nn.Module):
    def __init__(self, device, src_vocab:int, tgt_vocab:int, d_model:int = 512, h:int = 8, d_ff:int=2048, dropout:float=0.1, N:int=6):
        super(Transformer, self).__init__()
        c = copy.deepcopy
        print(f"dropout={dropout}")
        
        # embedding + positional encoding
        src_embedding = Embeddings(src_vocab, d_model).to(device)
        tgt_embedding = Embeddings(tgt_vocab, d_model).to(device)
        positional_encoder = PositionalEncoder(device, d_model, d_ff, dropout).to(device)
        self.src_embed = nn.Sequential(src_embedding, c(positional_encoder)).to(device)
        self.tgt_embed = nn.Sequential(tgt_embedding, c(positional_encoder)).to(device)

        # encoder + decoder
        attention = MultiHeadAttention(d_model, h, dropout).to(device)
        ffn = PositionwiseFeedForward(d_model, d_ff, dropout).to(device)
        self.encoder = Encoder(EncoderLayer(d_model, c(attention), c(ffn).to(device), dropout), N).to(device)
        self.decoder = Decoder(DecoderLayer(d_model, c(attention), c(attention), c(ffn), dropout).to(device), N).to(device)
        
        # generator
        self.generator = Generator(d_model, tgt_vocab).to(device)
        
        # This was important from their code.
        # Initialize parameters with Glorot / fan_avg.
        # Paper title: Understanding the difficulty of training deep feedforward neural networks Xavier
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, m, tgt, src_mask, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), m, tgt_mask, src_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        print(f"+TRANSFORMER:")
        # m = self.encoder(self.src_embed(src), src_mask)
        m = self.encode(src, src_mask)
        # output_trans = self.decoder(self.tgt_embed(tgt), m, tgt_mask, src_mask)
        output_trans = self.decode(m, tgt, src_mask, tgt_mask)
        print(f"-TRANSFORMER:output shape:{output_trans.shape}")
        return output_trans

In [ ]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape), k=1).astype('uint8')
    return torch.from_numpy(subsequent_mask) == 0

In [ ]:
class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(in_features=d_model, out_features=vocab)
    
    def forward(self, x):
        print(f"+GENERATOR:x.shape={x.shape}")
        proj_x = self.proj(x)
        g_out = F.log_softmax(proj_x, dim=-1)
        print(f"-GENERATOR:g_out.shape={g_out.shape}")
        return g_out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
src_vocab, tgt_vocab = 1024, 1024
batch_size, seq_len,  d_model, h , d_ff, dropout, N = 1, 10, 512, 8, 2048, 0.1, 2

# src should be the tokenized sentences: (batch_size, seq_len)
src = torch.randint(0, src_vocab, (batch_size, seq_len)).to(device)
src_mask = torch.ones(batch_size, 1, seq_len).to(device)  # Simple mask (all ones = no masking)
print(f"src shape:{src.shape} src_mask shape:{src_mask.shape}")

Token_BOS = 2
tgt = torch.ones(1, 1).fill_(Token_BOS).type_as(src.data)
# tgt_mask = subsequent_mask(tgt.size(1)).type_as(src.data)
# print(f"tgt shape:{tgt.shape} tgt mask shape:{tgt_mask.shape}")

model = Transformer(device, src_vocab, tgt_vocab, d_model, h, d_ff, dropout, N).to(device)

# y = model(src, tgt, src_mask, tgt_mask)
max_output_len = 20
# m = model.encode(src, src_mask)
# out = model.decode(m, tgt, src_mask, tgt_mask)
# print(f"out shape: {out.shape} - {out[:, -1].shape}")

# Autoregressive decoding loop
max_len = 20
for i in range(max_len):
    print(f"Step {i}")
    print(f"tgt shape = {tgt.shape}")
    # Create mask for current target sequence
    tgt_mask = subsequent_mask(tgt.size(1)).to(device)
    print(f"\ttgt_mask:{tgt_mask.shape}\n{tgt_mask}")

    # Decode
    out = model.decode(m, tgt, src_mask, tgt_mask)
    # out shape: (1, current_tgt_len, 512)
    print(f"\tout:{out.shape}")

    # Get last token's output
    # out[:, -1] equal to x[:,-1,:]
    last_token_features = out[:, -1]  # Shape: (1, 512)
    print(f"\tlast_token_features:{last_token_features.shape}")

    # Project to vocabulary (you'd need a generator/linear layer)
    HAS_GENERATOR = True
    if HAS_GENERATOR:
        next_token_logits = model.generator(last_token_features)  # (1, vocab_size)
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)  # (1, 1)
    else:
        # For demo, just use random token
        next_token = torch.randint(0, tgt_vocab, (1, 1)).to(device)
    
    # Append to target sequence
    tgt = torch.cat([tgt, next_token], dim=1)
    # print(f"Step {i+1}: tgt shape = {tgt.shape}")
    
    # Stop if EOS token (e.g., token 3)
    if next_token.item() == 3:
        break

print(f"Final generated sequence: {tgt}")



src shape:torch.Size([1, 10]) src_mask shape:torch.Size([1, 1, 10])
dropout=0.1
pe initial shape: torch.Size([2048, 512])
position shape: torch.Size([2048, 1])
div_term shape: torch.Size([1, 256])
pe_pos shape: torch.Size([2048, 256])
pe final shape: torch.Size([1, 2048, 512])
Step 0
tgt shape = torch.Size([1, 1])
	tgt_mask:torch.Size([1, 1, 1])
tensor([[[True]]], device='cuda:0')
[PE]: x shape: torch.Size([1, 1, 512])
+DECODER:x shape:torch.Size([1, 1, 512]) memory shape:torch.Size([1, 10, 512]) tgt_mask shape:torch.Size([1, 1, 1]) src_mask shape:torch.Size([1, 1, 10])
+DC_LAYER:x.shape=torch.Size([1, 1, 512]), memory.shape=torch.Size([1, 10, 512])
+RC: x.shape=torch.Size([1, 1, 512])
+LN x shape:torch.Size([1, 1, 512])
	LN: mean shape: torch.Size([1, 1, 1])
	LN: std shape: torch.Size([1, 1, 1])
-LN output shape:torch.Size([1, 1, 512])
+MHA:[1, 1, 512]
+AT: q/k/v shapes: torch.Size([1, 8, 1, 64])
	scores shape: torch.Size([1, 8, 1, 1])
	attn shape: torch.Size([1, 8, 1, 1])
-AT: output

In [24]:
x = torch.randint(0, src_vocab, (2, 3, 4))
print(f"x shape: {x.shape}\n {x}")
x_last = x[:, -1]
print(f"x_last shape: {x_last.shape}\n {x_last}")
x_last_2 = x[:,-1,:]
print(f"x_last_2 shape: {x_last_2.shape}\n {x_last_2}")

x shape: torch.Size([2, 3, 4])
 tensor([[[ 589,  550,  122,  231],
         [ 408,  705,  192,  902],
         [ 893,  321,  420,  231]],

        [[ 525,  824,   86,  369],
         [1005,  747,  813,  178],
         [ 913,  812,  601,  488]]])
x_last shape: torch.Size([2, 4])
 tensor([[893, 321, 420, 231],
        [913, 812, 601, 488]])
x_last_2 shape: torch.Size([2, 4])
 tensor([[893, 321, 420, 231],
        [913, 812, 601, 488]])
